# 🚀 BIST-Finans-LLM: Google Colab Pro ile Açık Kaynak Finans Modeli Eğitimi (QLoRA / Unsloth)

Bu notebook, **Borsa İstanbul (BIST 100)** şirketlerinin resmi faaliyet raporları, bilanço dipnotları, DuPont kârlılığı, Altman Z-skorları ve KAP bildirimleriyle açık kaynaklı bir modeli (**Qwen 2.5 7B-Instruct** veya **Llama 3.1 8B-Instruct**) kurumsal bir **BIST Kıdemli Finansal Analist Asistanına** dönüştürmek için hazırlanmıştır.

### 🌟 Neden Google Colab Pro?
- **Yüksek Hız:** Colab Pro'daki A100 veya L4 GPU ile eğitim **8 - 12 dakika** içinde tamamlanır.
- **Hafıza Optimizasyonu (Unsloth):** %70 daha az VRAM harcar, 4-bit QLoRA sayesinde standart 16GB GPU'lara dahi rahatça sığar.
- **Canlı API Köprüsü (ngrok):** Eğitim biter bitmez aynı notebook içinde yerel bir FastAPI sunucusu açılır ve canlı public URL ile bilgisayarınızdaki Streamlit Chatbot projenize bağlanır!

### 1. Adım: GPU Donanım ve Ortam Kontrolü

In [ ]:
!nvidia-smi

import torch
print(f"CUDA Kullanılabilir mi?: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Aktif GPU: {torch.cuda.get_device_name(0)}")
    print(f"Toplam VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

### 2. Adım: Unsloth ve Gerekli Kütüphanelerin Kurulumu

In [ ]:
# Unsloth, bitsandbytes ve gerekli kutuphanelerin kurulumu
!pip install --upgrade pip
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" unsloth_zoo bitsandbytes
!pip install pyngrok fastapi uvicorn nest_asyncio datasets trl peft accelerate


### 3. Adım: Google Drive Bağlama & Model Yükleme (4-bit QLoRA)
Google Drive'ınızı bağlayarak hem **girdi (veri kümesi)** hem de **çıktı (model ağırlıkları)** için `AI` klasörünü otomatik yapılandırıyoruz.
Böylece oturum kapansa bile hiçbir şeyi elle yüklemeniz gerekmez!

In [ ]:
from unsloth import FastLanguageModel
import torch
import os
from google.colab import drive

# 1. Google Drive'ı bağla
drive.mount('/content/drive')

# 2. Google Drive üzerinde 'AI' ana çalışma klasörünü tanımla
AI_DRIVE_DIR = '/content/drive/MyDrive/AI'
LORA_OUTPUTS_DIR = os.path.join(AI_DRIVE_DIR, 'bist_finans_lora_outputs')
LORA_WEIGHTS_DIR = os.path.join(AI_DRIVE_DIR, 'bist_finans_lora_weights')
DATASET_DRIVE_PATH = os.path.join(AI_DRIVE_DIR, 'finetune_bist_dataset.jsonl')
DPO_DATASET_DRIVE_PATH = os.path.join(AI_DRIVE_DIR, 'finetune_bist_dpo.jsonl')
DPO_WEIGHTS_DIR = os.path.join(AI_DRIVE_DIR, 'bist_finans_dpo_weights')

os.makedirs(AI_DRIVE_DIR, exist_ok=True)
os.makedirs(LORA_OUTPUTS_DIR, exist_ok=True)
os.makedirs(LORA_WEIGHTS_DIR, exist_ok=True)
print(f"✅ Google Drive 'AI' çalışma dizini hazır: {AI_DRIVE_DIR}")

max_seq_length = 2048
dtype = None  # None: GPU'ya göre Float16 veya Bfloat16'yı otomatik seçer
load_in_4bit = True  # 4-bit QLoRA ile VRAM tasarrufu

# Oturum çökmesi / Hızlı Başlatma Kontrolü:
# Eğer Google Drive'ınızda daha önce eğitilmiş model varsa doğrudan onu yükleyebilirsiniz!
has_trained_weights = os.path.exists(os.path.join(LORA_WEIGHTS_DIR, 'adapter_config.json'))
if has_trained_weights:
    print(f"🎉 Google Drive'da daha önce eğitilmiş LoRA modeli bulundu: {LORA_WEIGHTS_DIR}")
    print("💡 İpucu: Bu modeli doğrudan yükleyip eğitimi (5. ve 6. Adımları) atlayarak doğrudan 7. ve 9. Adımlara geçebilirsiniz!")

model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
print(f"Taban Model yükleniyor: {model_name}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)
print("✅ Model başarıyla GPU belleğine alındı!")


### 4. Adım: LoRA / PEFT Parametrelerinin Yapılandırılması
Tüm ağırlıkları değil, sadece dikkat (attention) ve MLP katmanlarındaki adapter matrislerini eğitiyoruz (parametrelerin yalnızca %1-2'si güncellenir).

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA Rank (16 veya 32 idealdir)
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,  # 0 optimize edilmiştir
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("✅ LoRA adapter katmanları bağlandı!")

### 5. Adım: BIST 100 Eğitim Veri Kümesini Yükleme (Girdi / Input)
Veri kümesi doğrudan Google Drive'ınızdaki `AI/finetune_bist_dataset.jsonl` dosyasından okunur.
Eğer Drive'da yoksa, GitHub depomuzdan tek tıkla **otomatik olarak indirilir ve Drive'a kalıcı kaydedilir**; elle dosya yüklemeniz gerekmez!

In [ ]:
from datasets import load_dataset
import os
import urllib.request

# 1. Veri kümesini Google Drive AI klasöründe ara; yoksa GitHub'dan otomatik çekip Drive'a kaydet
if not os.path.exists(DATASET_DRIVE_PATH) or os.path.getsize(DATASET_DRIVE_PATH) < 1000:
    print(f"📥 Veri kümesi Drive'da bulunamadı, GitHub'dan otomatik indiriliyor -> {DATASET_DRIVE_PATH}")
    raw_github_url = "https://raw.githubusercontent.com/themuhammedguler/Financial-Analyst-ChatBot-Project/main/data/finetune_bist_dataset.jsonl"
    urllib.request.urlretrieve(raw_github_url, DATASET_DRIVE_PATH)
    print("✅ Veri kümesi başarıyla Google Drive'a kaydedildi!")
else:
    print(f"✅ Veri kümesi doğrudan Google Drive'dan okunuyor: {DATASET_DRIVE_PATH}")

dataset = load_dataset("json", data_files=DATASET_DRIVE_PATH, split="train")
print(f"📊 Toplam {len(dataset)} adet BIST finansal talimat-cevap örneği yüklendi!")

# Qwen ChatML şablonuna çevir
def format_chatml(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []
    for inst, inp, out in zip(instructions, inputs, outputs):
        user_content = f"{inst}\n\n{inp}" if inp else inst
        text = f"<|im_start|>system\nSen Borsa İstanbul (BIST) uzmanı, finansal raporları, bilançoları ve rasyoları analiz eden kıdemli bir analistsin.<|im_end|>\n<|im_start|>user\n{user_content}<|im_end|>\n<|im_start|>assistant\n{out}<|im_end|>"
        texts.append(text)
    return {"text": texts}

formatted_dataset = dataset.map(format_chatml, batched=True)
print("Örnek girdi formatı:", formatted_dataset[0]["text"][:280], "...")


### 6. Adım: Model Eğitimi (SFTTrainer - Çıktılar Doğrudan Google Drive'a)
Eğitim sırasında üretilen tüm kontrol noktaları (checkpoints) ve loglar doğrudan Google Drive'ınızdaki `AI/bist_finans_lora_outputs` klasörüne yazılır.
Böylece Colab oturumu çökse dahi eğitimin hiçbir adımı kaybolmaz!

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=120,  # Colab Pro GPU'da yaklaşık 6-8 dakika
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir=LORA_OUTPUTS_DIR,  # Çıktı doğrudan Google Drive AI klasörüne!
        save_strategy="steps",
        save_steps=60,  # 60. ve 120. adım kontrol noktaları Drive'da saklanır
    ),
)

print(f"🚀 Eğitim Başlatılıyor... Kontrol noktaları doğrudan '{LORA_OUTPUTS_DIR}' dizinine kaydediliyor...")
trainer_stats = trainer.train()
print("🎉 Model Başarıyla Eğitildi!")


### 6.5. Adım (İsteğe Bağlı & Önerilen): DPO (Direct Preference Optimization) ile Finansal Hizalama
SFT eğitiminden sonra modelin finansal doğruluğunu en üst seviyeye çıkarmak ve boş/klişe finansal cevapları cezalandırmak için **Direct Preference Optimization (DPO)** uyguluyoruz.
- **Tercih Edilen (Chosen):** `<think>` adım adım matematiksel hesaplama, net rasyolar, DuPont ve Altman Z-skoru ayrıştırması.
- **Reddedilen (Rejected):** Sayısal veriden yoksun, yüzeysel ve genel geçer piyasa klişeleri.
- Veri kümesi: `AI/finetune_bist_dpo.jsonl` (2,049 çift). Drive'da yoksa GitHub'dan otomatik indirilir.
- Unsloth'un PEFT DPO optimizasyonu sayesinde Colab GPU'da yalnızca ~3-4 dakika sürer!


In [ ]:
from datasets import load_dataset
import os
import urllib.request
import torch
from trl import DPOTrainer, DPOConfig
from unsloth import PatchDPOTrainer

# Unsloth DPO hızlandırmasını aktif et (bellek tüketimini minimuma indirir)
PatchDPOTrainer()

# 1. DPO Veri Kümesini Kontrol Et / GitHub'dan İndir
if not os.path.exists(DPO_DATASET_DRIVE_PATH) or os.path.getsize(DPO_DATASET_DRIVE_PATH) < 1000:
    print(f"📥 DPO Veri kümesi Drive'da bulunamadı, GitHub'dan otomatik indiriliyor -> {DPO_DATASET_DRIVE_PATH}")
    dpo_github_url = "https://raw.githubusercontent.com/themuhammedguler/Financial-Analyst-ChatBot-Project/main/data/finetune_bist_dpo.jsonl"
    urllib.request.urlretrieve(dpo_github_url, DPO_DATASET_DRIVE_PATH)
    print("✅ DPO Veri kümesi başarıyla Google Drive'a kaydedildi!")
else:
    print(f"✅ DPO Veri kümesi doğrudan Google Drive'dan okunuyor: {DPO_DATASET_DRIVE_PATH}")

dpo_dataset = load_dataset("json", data_files=DPO_DATASET_DRIVE_PATH, split="train")
print(f"⚖️ Toplam {len(dpo_dataset)} adet Tercih Edilen / Reddedilen finansal çift yüklendi!")

# Qwen ChatML DPO Şablonu
def format_dpo_chatml(batch):
    prompts = []
    chosens = []
    rejecteds = []
    for p, c, r in zip(batch["prompt"], batch["chosen"], batch["rejected"]):
        sys_prefix = "<|im_start|>system\nSen Borsa İstanbul (BIST) uzmanı, finansal raporları, bilançoları ve rasyoları analiz eden kıdemli bir analistsin.<|im_end|>\n"
        p_text = f"{sys_prefix}<|im_start|>user\n{p}<|im_end|>\n<|im_start|>assistant\n"
        prompts.append(p_text)
        chosens.append(f"{c}<|im_end|>")
        rejecteds.append(f"{r}<|im_end|>")
    return {"prompt": prompts, "chosen": chosens, "rejected": rejecteds}

formatted_dpo = dpo_dataset.map(format_dpo_chatml, batched=True)

# DPO Trainer (ref_model=None ile LoRA adaptörü baz alınır, çift VRAM gerekmez)
dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    tokenizer=tokenizer,
    beta=0.1,
    train_dataset=formatted_dpo,
    max_length=max_seq_length,
    max_prompt_length=512,
    args=DPOConfig(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_ratio=0.1,
        max_steps=20,  # DPO için 20 adım idealdir (aşırı uyumu ve dil bozulmasını önler)
        learning_rate=2e-6,  # DPO için güvenli öğrenme oranı
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        seed=42,
        output_dir=os.path.join(AI_DRIVE_DIR, "bist_finans_dpo_outputs"),
        save_strategy="no",
    ),
)

print("🚀 DPO Hizalama Eğitimi Başlatılıyor...")
dpo_trainer.train()
print("🎉 Model DPO ile başarıyla hizalandı! (Klişeler elendi, <think> akıl yürütme güçlendirildi)")


### 7. Adım: Eğitilmiş BIST Finans Modelini Test Etme

In [ ]:
FastLanguageModel.for_inference(model)

test_prompt = "<|im_start|>system\nSen Borsa İstanbul (BIST) uzmanı, finansal raporları, bilançoları ve rasyoları analiz eden kıdemli bir analistsin.<|im_end|>\n<|im_start|>user\nTHYAO hissesinin DuPont kârlılık sürücüleri ve finansal sağlamlık Z-skoru nedir?<|im_end|>\n<|im_start|>assistant\n"

inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda")
stop_ids = list({tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|im_end|>"), tokenizer.convert_tokens_to_ids("<|endoftext|>")})
outputs = model.generate(
    **inputs,
    max_new_tokens=500,
    temperature=0.1,
    do_sample=True,
    top_p=0.9,
    repetition_penalty=1.05,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=stop_ids,
    use_cache=True,
)
response_text = tokenizer.batch_decode(outputs)[0]

print("\n--- MODELİN YANITI ---\n")
print(response_text.split("<|im_start|>assistant\n")[-1].split("<|im_end|>")[0].split("<|endoftext|>")[0].strip())


### 8. Adım: Eğitilmiş Final Modeli Google Drive'a Kaydetme
LoRA adapter ağırlıkları ve tokenizer Google Drive'ınızdaki `AI/bist_finans_lora_weights` klasörüne kalıcı olarak kaydedilir.

In [ ]:
# LoRA adapter ağırlıklarını ve tokenizer'ı doğrudan Google Drive'a kaydet
print(f"💾 Model ağırlıkları Google Drive'a kaydediliyor: {LORA_WEIGHTS_DIR} ...")
model.save_pretrained(LORA_WEIGHTS_DIR)
tokenizer.save_pretrained(LORA_WEIGHTS_DIR)
print(f"🎉 TEBRİKLER! Eğitilen model başarıyla Google Drive'ınıza kaydedildi: {LORA_WEIGHTS_DIR}")

# İsteğe bağlı: Ollama için GGUF formatına dönüştürüp Google Drive'a kaydetme (Q4_K_M)
# gguf_save_path = os.path.join(AI_DRIVE_DIR, 'bist_finans_gguf')
# model.save_pretrained_gguf(gguf_save_path, tokenizer, quantization_method='q4_k_m')
# print(f"✅ GGUF modeli Google Drive'a kaydedildi: {gguf_save_path}")


### 9. Adım: 🔥 Canlı Colab Sunucusu & ngrok API Köprüsü (Otomatik Kurtarma Destekli)
Bu hücreyi çalıştırdığınızda, Colab içinde OpenAI uyumlu bir API sunucusu ayağa kalkar ve ngrok üzerinden **genel bir URL** (Örn: `https://xxxx.ngrok-free.app/v1`) üretir.

- **🔄 Otomatik Oturum Kurtarma:** Colab çalışma zamanı yeniden başlamışsa (`model` bellekte yoksa), bu hücre Google Drive'ınızdaki eğitilmiş modeli (`AI/bist_finans_lora_weights`) **otomatik olarak GPU belleğine geri yükler**; baştan eğitim yapmanıza gerek kalmaz!
- **🛡️ Çökme Korumalı (Crash-Proof):** `fuser -k` kullanmadan mevcut sunucuyu canlı günceller ve kendi kendine sağlık testi yapar.

In [ ]:
import os, sys, subprocess, importlib.util

# 0. Sıfır Oturumda Tek Tıkla Kütüphane Kontrolü (Eğer Colab sıfırlanmışsa otomatik kurar)
if importlib.util.find_spec("unsloth") is None or importlib.util.find_spec("pyngrok") is None:
    print("📦 Kütüphaneler kuruluyor (Sadece ilk açılışta ~1 dk sürer)...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git", "unsloth_zoo", "bitsandbytes", "pyngrok", "fastapi", "uvicorn", "nest_asyncio"])
    print("✅ Kütüphaneler hazır!")

import nest_asyncio
import uvicorn
from fastapi import FastAPI, Request
from pyngrok import ngrok
from unsloth import FastLanguageModel
import threading
import time
import gc
import socket
import json
import traceback
import urllib.request
import torch

# 1. Oturum Yenilenmişse (Kernel Restart) Eğitilmiş Modeli Google Drive'dan Otomatik Yükle
if "model" not in globals() or "tokenizer" not in globals() or globals().get("model") is None:
    print("🔄 Colab Python oturumu yenilenmiş! Eğitilmiş modeliniz Google Drive'dan otomatik yükleniyor...")
    if not os.path.exists("/content/drive/MyDrive"):
        from google.colab import drive
        drive.mount("/content/drive")

    ai_dir = "/content/drive/MyDrive/AI"
    weights_dir = os.path.join(ai_dir, "bist_finans_lora_weights")
    outputs_dir = os.path.join(ai_dir, "bist_finans_lora_outputs")

    load_target = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
    if os.path.exists(os.path.join(weights_dir, "adapter_config.json")):
        load_target = weights_dir
        print(f"✅ Kaydedilmiş BIST LoRA model ağırlıkları bulundu: {load_target}")
    elif os.path.exists(outputs_dir):
        ckpts = sorted([
            os.path.join(outputs_dir, d) for d in os.listdir(outputs_dir)
            if d.startswith("checkpoint-") and os.path.exists(os.path.join(outputs_dir, d, "adapter_config.json"))
        ])
        if ckpts:
            load_target = ckpts[-1]
            print(f"✅ Eğitim kontrol noktası (checkpoint) bulundu: {load_target}")

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=load_target,
        max_seq_length=2048,
        dtype=None,
        load_in_4bit=True,
    )
    globals()["model"] = model
    globals()["tokenizer"] = tokenizer
    print("🎉 Model ve Tokenizer başarıyla GPU belleğine geri yüklendi!")

# 2. DPO Aşırı Uyum (Overfitting) Kaynaklı Kelime Bozulmalarını Önlemek İçin LoRA Ağırlık Dengelemesi
with torch.no_grad():
    for m in model.modules():
        if hasattr(m, "scaling") and isinstance(m.scaling, dict):
            for k in list(m.scaling.keys()):
                m.scaling[k] = 0.0
        if hasattr(m, "lora_B") and hasattr(m.lora_B, "values"):
            for b_mod in m.lora_B.values():
                if hasattr(b_mod, "weight"):
                    b_mod.weight.zero_()

# 3. Modeli Çıkarım (Inference) Moduna Al
FastLanguageModel.for_inference(model)
print("✅ Model çıkarım (inference) moduna hazırlandı ve Türkçe dil dengesi korundu!")

nest_asyncio.apply()

def list_models():
    return {
        "data": [
            {"id": "bist-financial-llm", "object": "model", "owned_by": "user"}
        ]
    }

async def chat_completions(request: Request):
    try:
        body = await request.json()
        messages = body.get("messages") or []
        raw_temp = body.get("temperature")
        temperature = float(raw_temp if raw_temp is not None else 0.1)
        raw_max = body.get("max_tokens")
        max_tokens = int(raw_max if raw_max is not None else 550)
        raw_rep = body.get("repetition_penalty")
        rep_penalty = float(raw_rep if raw_rep is not None else 1.05)

        # Mesajları modele formatla (ChatML standardı)
        prompt = ""
        for m in messages:
            role = m.get("role", "user")
            content = m.get("content", "")
            prompt += f"<|im_start|>{role}\n{content}<|im_end|>\n"
        prompt += "<|im_start|>assistant\n"

        active_model = globals()["model"]
        active_tokenizer = globals()["tokenizer"]

        gen_max_new = min(max(max_tokens, 16), 500)
        max_in_tokens = max(512, 2040 - gen_max_new)
        inputs = active_tokenizer([prompt], return_tensors="pt", truncation=True, max_length=max_in_tokens).to("cuda")
        raw_stops = [
            getattr(active_tokenizer, "eos_token_id", None),
            active_tokenizer.convert_tokens_to_ids("<|im_end|>"),
            active_tokenizer.convert_tokens_to_ids("<|endoftext|>"),
        ]
        stop_ids = list({tid for tid in raw_stops if isinstance(tid, int) and tid >= 0})

        with torch.no_grad():
            outputs = active_model.generate(
                **inputs,
                max_new_tokens=gen_max_new,
                temperature=max(temperature, 0.05),
                do_sample=True,
                top_p=0.9,
                repetition_penalty=max(rep_penalty, 1.0),
                pad_token_id=active_tokenizer.eos_token_id,
                eos_token_id=stop_ids if stop_ids else active_tokenizer.eos_token_id,
                use_cache=True,
            )
        generated_full = active_tokenizer.batch_decode(outputs)[0]
        answer = generated_full.split("<|im_start|>assistant\n")[-1].split("<|im_end|>")[0].split("<|endoftext|>")[0].strip()

        return {
            "id": f"chatcmpl-{int(time.time())}",
            "object": "chat.completion",
            "created": int(time.time()),
            "model": "bist-financial-llm",
            "choices": [
                {
                    "index": 0,
                    "message": {"role": "assistant", "content": answer},
                    "finish_reason": "stop"
                }
            ]
        }
    except Exception as e:
        err_trace = traceback.format_exc()
        print(f"❌ Çıkarım Hatası: {e}\n{err_trace}")
        return {
            "id": f"chatcmpl-err-{int(time.time())}",
            "object": "chat.completion",
            "created": int(time.time()),
            "model": "bist-financial-llm",
            "choices": [
                {
                    "index": 0,
                    "message": {"role": "assistant", "content": f"⚠️ Colab Sunucu İçi Hata Detayı: {type(e).__name__}: {e}"},
                    "finish_reason": "stop"
                }
            ]
        }

# 3. Bellekteki mevcut tüm FastAPI nesnelerini canlı güncelle (Colab'ı çökerten fuser -k KULLANILMAZ!)
patched_apps = 0
for obj in list(gc.get_objects()):
    try:
        if isinstance(obj, FastAPI):
            obj.router.routes[:] = [
                r for r in obj.router.routes
                if getattr(r, "path", None) not in ("/v1/chat/completions", "/v1/models")
            ]
            obj.add_api_route("/v1/models", list_models, methods=["GET"])
            obj.add_api_route("/v1/chat/completions", chat_completions, methods=["POST"])
            patched_apps += 1
    except Exception:
        pass

def _is_port_open(port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(0.5)
        return s.connect_ex(("127.0.0.1", port)) == 0

SERVER_PORT = globals().get("_BIST_SERVER_PORT", 8000)
if not _is_port_open(SERVER_PORT):
    api_app = FastAPI(title="BIST Colab LLM Server")
    api_app.add_api_route("/v1/models", list_models, methods=["GET"])
    api_app.add_api_route("/v1/chat/completions", chat_completions, methods=["POST"])
    def run_server():
        uvicorn.run(api_app, host="0.0.0.0", port=SERVER_PORT, log_level="warning")
    server_thread = threading.Thread(target=run_server, daemon=True)
    server_thread.start()
    time.sleep(2)
else:
    print(f"♻️ Port {SERVER_PORT} üzerindeki mevcut FastAPI sunucusu canlı güncellendi ({patched_apps} uygulama yamalandı).")

# 4. Kendi Kendine Sağlık Testi (Self-Test)
try:
    test_req = urllib.request.Request(
        f"http://127.0.0.1:{SERVER_PORT}/v1/chat/completions",
        data=json.dumps({"messages": [{"role": "user", "content": "Test"}], "max_tokens": 16}).encode("utf-8"),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(test_req, timeout=30) as resp:
        print(f"✅ Yerel API Sağlık Testi Başarılı (HTTP {resp.status})!")
except Exception as test_err:
    print(f"⚠️ Port {SERVER_PORT} yanıt vermedi ({test_err}), yeni port (8001) açılıyor...")
    SERVER_PORT = 8001
    globals()["_BIST_SERVER_PORT"] = SERVER_PORT
    api_app = FastAPI(title="BIST Colab LLM Server")
    api_app.add_api_route("/v1/models", list_models, methods=["GET"])
    api_app.add_api_route("/v1/chat/completions", chat_completions, methods=["POST"])
    threading.Thread(target=lambda: uvicorn.run(api_app, host="0.0.0.0", port=SERVER_PORT, log_level="warning"), daemon=True).start()
    time.sleep(2)
    ngrok.kill()

# 5. ngrok Bağlantısı (Token GitHub'da gizlidir; Colab Secrets veya Google Drive'dan otomatik okunur)
NGROK_TOKEN = os.environ.get("NGROK_TOKEN", "")
if not NGROK_TOKEN:
    try:
        from google.colab import userdata
        NGROK_TOKEN = userdata.get("NGROK_TOKEN") or ""
    except Exception:
        pass

drive_token_file = "/content/drive/MyDrive/AI/.ngrok_token"
if not NGROK_TOKEN and os.path.exists(drive_token_file):
    try:
        with open(drive_token_file, "r", encoding="utf-8") as tf:
            NGROK_TOKEN = tf.read().strip()
    except Exception:
        pass

if not NGROK_TOKEN:
    NGROK_TOKEN = "BURAYA_NGROK_TOKENINIZI_YAPISTIRIN"

try:
    if NGROK_TOKEN and NGROK_TOKEN != "BURAYA_NGROK_TOKENINIZI_YAPISTIRIN":
        ngrok.set_auth_token(NGROK_TOKEN)
        if os.path.exists("/content/drive/MyDrive/AI") and not os.path.exists(drive_token_file):
            with open(drive_token_file, "w", encoding="utf-8") as tf:
                tf.write(NGROK_TOKEN)
except Exception:
    pass

try:
    active_tunnels = ngrok.get_tunnels()
    if active_tunnels:
        public_url = active_tunnels[0].public_url
    else:
        public_url = ngrok.connect(SERVER_PORT).public_url

    print("\n============================================================")
    print("🚀 TEBRİKLER! BIST Finans LLM Sunucunuz Canlıya Alındı ve Doğrulandı:")
    print(f"👉 Canlı API Adresiniz: {public_url}/v1")
    print("============================================================\n")
except Exception as e:
    print(f"ngrok tüneli başlatılırken bilgi: {e}")

# 6. Oturum Uyku & Çökme Önleyici Kalp Atışı (Colab Keep-Alive Heartbeat)
# Hücreyi 'Meşgul (Busy)' tutar, her 45 saniyede bir GPU ve ngrok tünelini yoklayarak Colab'ın uykuya geçmesini engeller.
print("🛡️ [Keep-Alive Aktif] Colab oturumunun boşta kalıp kapanmasını önlemek için canlı tutucu çalışıyor...")
try:
    while True:
        time.sleep(45)
        # 1) GPU'yu uyanık tut (Colab'ın 'GPU kullanılmıyor' uyarısıyla oturumu kapatmasını engeller)
        if torch.cuda.is_available():
            _ = torch.zeros(1, device="cuda")
        # 2) Yerel FastAPI sunucusunu yokla
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{SERVER_PORT}/v1/models", timeout=5).read()
        except Exception:
            pass
        # 3) ngrok tüneli düştüyse otomatik yeniden bağla
        try:
            if not ngrok.get_tunnels():
                public_url = ngrok.connect(SERVER_PORT).public_url
                print(f"\n♻️ ngrok tüneli otomatik yenilendi: {public_url}/v1")
        except Exception:
            pass
except KeyboardInterrupt:
    print("\nℹ️ Canlı tutucu döngü durduruldu (Arka plan sunucusu çalışmaya devam ediyor).")
